# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model('groq:qwen/qwen3-32b')

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

### Message Size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message based summarization
agent = create_agent(
    model=model,
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
        model=model,
        trigger=("messages", 10),
        keep=("messages", 4)
        )
    ]
)

In [3]:
# Run with thread id
config = {"configurable":{"thread_id":"test-1"}}

In [4]:
questions = [
    "What is 2+2?",
    "What is 100/45?",
    "What is 5*25?",
    "What is 12-3?",
    "What is 120*10?",
    "What is 12-3*10?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='31c293e7-e11e-4288-a610-779aec0aba9c'), AIMessage(content='<think>\nOkay, the user is asking "What is 2+2?" That seems straightforward. Let me make sure I understand the question correctly. They want to know the sum of two and two.\n\nFirst, I should confirm that this is a basic arithmetic question. Adding 2 and 2 together is one of the simplest operations in math. The answer is 4, but maybe I should explain it step by step just in case.\n\nLet me break it down. If you have two objects, like apples, and you add two more apples, how many do you have in total? You start with two, then add another two. Counting them: one, two, three, four. So that\'s four.\n\nI should also consider if there\'s any chance the user is being tricky or if there\'s a context where the answer isn\'t 4. For example, in some non-standard arithmetic systems or in certain wordplay scenarios, 2+2 might not be

### Token Size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

@tool
def search_hotels(city:str) -> str :
    ''''Search hotels - return long response to use more tokens'''
    return f"""hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",800),
            keep=("tokens",300)
        )
    ]
)

In [21]:
config = {"configurable":{"thread_id":"test-1"}}

In [22]:
#Token counter (approximate)

def count_tokens (messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4   # 4 chars = 1 token

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~178 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='da5315b6-91d4-4fa5-9316-8700cbd49d39'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The city here is Paris. I need to make sure the function is called correctly. The parameters should be {"city": "Paris"}. I\'ll generate the tool call with that information.\n', 'tool_calls': [{'id': '2t5sef7n6', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 155, 'total_tokens': 250, 'completion_time': 0.149124779, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.012102214, 'prompt_tokens_details': None, 'queue_time': 0.451478155, 'total_time': 0.161226993}, 'model_nam

### Fraction

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

@tool
def search_hotels(city:str) -> str :
    '''Search hotels'''
    return f"""hotels in {city}: 1. Grand Hotel - 5 star, $350/night, spa, pool, gym"""

agent = create_agent(
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction",0.005),  # 0.5% = ~640 tokens
            keep=("fraction",0.002)      # 0.2% = ~256 tokens 
        )
    ]
)

In [3]:
config = {"configurable":{"thread_id":"test-1"}}

In [4]:
#Token counter (approximate)

def count_tokens (messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4   # 4 chars = 1 token

In [5]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  #gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~72 tokens (0.0562%), 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='0bbc53d1-c96b-4de1-a8d7-a1bbfb74cf9f'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The required parameter is city, and it\'s a string. Since the user mentioned Paris, I need to call this function with "Paris" as the city argument. I\'ll make sure the JSON is correctly formatted with the city name in quotes. No other parameters are needed here. Alright, time to structure the tool call properly.\n', 'tool_calls': [{'id': 'hqzhs4jgh', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 147, 'total_tokens': 270, 'completion_time': 0.188397082, 'completion_tokens_details': 